# Step 11 — Feature Selection Workbench

This notebook explores which features matter most for regime classification
and whether the `clustering_features` list in `settings.yaml` can be pruned.

Key questions:
1. Which features carry the most information for distinguishing regimes?
2. How many features are needed for 90% cumulative importance?
3. What happens to clustering quality if we drop low-importance features?
4. Are there "dead" features contributing nearly zero information?

**Requires:** pipeline steps 1-5 (needs `current_regime.pkl` RF model).

## Setup + Load RF Model & Feature Importances (D9.1)

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from trading_crab_lib.config import load, setup_logging
from trading_crab_lib.runtime import RunConfig
from trading_crab_lib.checkpoints import CheckpointManager
from trading_crab_lib import DATA_DIR, OUTPUT_DIR, plotting

setup_logging("INFO")
log = logging.getLogger("11_feature_selection")
cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=False)
cm = CheckpointManager()

In [ ]:
# Load RF model importances and features
importances = None
features = None
kmeans_labels = None
clustering_features = cfg.get("features", {}).get("clustering_features", [])

# Feature importances from RF model
model_path = OUTPUT_DIR / "models" / "current_regime.pkl"
try:
    from trading_crab_lib.cluster_comparison import extract_rf_feature_importances
    importances = extract_rf_feature_importances(model_path)
    print(f"RF importances loaded: {len(importances)} features")
    print(f"Top 10:\n{importances.head(10).to_string()}")
except FileNotFoundError:
    print(f"RF model not found at {model_path}")
    print("Run: python run_pipeline.py --steps 1,2,3,4,5")
except Exception as e:
    print(f"Error loading RF model: {e}")

# Features (for re-clustering experiments)
try:
    features = cm.load("features")
    print(f"\nFeatures loaded: {features.shape}")
except Exception:
    print("Features checkpoint not found.")

# KMeans labels (for comparison baseline)
try:
    cluster_labels = cm.load("cluster_labels")
    if "balanced_cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["balanced_cluster"]
    elif "cluster" in cluster_labels.columns:
        kmeans_labels = cluster_labels["cluster"]
    print(f"KMeans labels loaded: {kmeans_labels.nunique()} regimes")
except Exception:
    print("Cluster labels not found.")

## Feature Importance Cumulative Curve (D9.2)

Cumulative importance as features are added — shows how many features
are needed to reach 90% or 95% of total importance. A steep curve means
a few features dominate; a flat curve means many features contribute equally.

In [ ]:
if importances is not None:
    plotting.plot_feature_selection_curve(
        importances, run_cfg,
        filename="11_feature_selection_curve.png",
    )

    # Summary statistics
    sorted_imp = importances.sort_values(ascending=False)
    cum = np.cumsum(sorted_imp.values) / sorted_imp.values.sum()
    n_90 = int(np.searchsorted(cum, 0.9)) + 1
    n_95 = int(np.searchsorted(cum, 0.95)) + 1
    print(f"Features for 90% importance: {n_90} of {len(importances)}")
    print(f"Features for 95% importance: {n_95} of {len(importances)}")
    print(f"Top feature: {sorted_imp.index[0]} ({sorted_imp.iloc[0]:.4f})")
    print(f"Bottom feature: {sorted_imp.index[-1]} ({sorted_imp.iloc[-1]:.6f})")
else:
    print("No importances available.")

## Recommended Feature Subset (D9.3)

Uses `recommend_clustering_features()` to intersect RF importances with
the current `clustering_features` list and suggest a leaner set.
Shows which current features would be kept vs dropped.

In [ ]:
if importances is not None and clustering_features:
    from trading_crab_lib.cluster_comparison import recommend_clustering_features

    recommended, comparison_df = recommend_clustering_features(
        importances, clustering_features, top_k=35
    )
    print(f"Recommended features: {len(recommended)} of {len(clustering_features)}")
    print(f"\nComparison table:")
    display(comparison_df)

    # Highlight what would be dropped
    dropped = comparison_df[~comparison_df["recommended"]]
    if not dropped.empty:
        print(f"\nFeatures that would be DROPPED ({len(dropped)}):")
        for _, row in dropped.iterrows():
            print(f"  {row.name}: importance={row.get('importance', 'N/A')}")
    else:
        print("\nAll clustering features are recommended (intersection < top_k).")
else:
    print("Importances or clustering_features not available.")

## What-If: Re-cluster with Top-35 Features (D9.4)

Compare clustering quality (silhouette score) when using all features
vs only the top-35 by RF importance. If the reduced set produces
comparable silhouette, the dropped features are noise for clustering.

In [ ]:
if features is not None and importances is not None and clustering_features:
    from trading_crab_lib.clustering import reduce_pca, evaluate_kmeans
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler

    n_comp = cfg.get("clustering", {}).get("n_pca_components", 5)
    balanced_k = cfg.get("clustering", {}).get("balanced_k", 5)

    # Full feature set
    full_cols = [c for c in clustering_features if c in features.columns]
    feat_full = features[full_cols].dropna()

    # Reduced feature set (top-35 from recommend_clustering_features)
    from trading_crab_lib.cluster_comparison import recommend_clustering_features
    recommended, _ = recommend_clustering_features(importances, clustering_features, top_k=35)
    reduced_cols = [c for c in recommended if c in features.columns]
    feat_reduced = features[reduced_cols].dropna()

    # Align to same index
    common_idx = feat_full.index.intersection(feat_reduced.index)
    feat_full = feat_full.loc[common_idx]
    feat_reduced = feat_reduced.loc[common_idx]

    results = []
    for label, feat_df in [("Full set", feat_full), ("Top-35", feat_reduced)]:
        pca_df, pca_obj, _ = reduce_pca(feat_df, n_components=min(n_comp, feat_df.shape[1]))
        X = StandardScaler().fit_transform(pca_df.values)
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=balanced_k, n_init=50, random_state=42)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        results.append({"Feature Set": label, "N Features": feat_df.shape[1],
                        "N Quarters": len(feat_df), "Silhouette": f"{sil:.4f}"})
        print(f"{label}: {feat_df.shape[1]} features, silhouette = {sil:.4f}")

    display(pd.DataFrame(results).set_index("Feature Set"))

    # Bar chart comparison
    fig, ax = plt.subplots(figsize=(6, 4))
    labels_bar = [r["Feature Set"] for r in results]
    sils = [float(r["Silhouette"]) for r in results]
    colors = [plotting.CUSTOM_COLORS[0], plotting.CUSTOM_COLORS[2]]
    ax.bar(labels_bar, sils, color=colors[:len(results)], alpha=0.8, width=0.5)
    ax.set_ylabel("Silhouette Score")
    ax.set_title(f"Clustering Quality: Full vs Reduced Feature Set (k={balanced_k})", fontsize=11)
    for i, v in enumerate(sils):
        ax.text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plotting._save_or_show(fig, "11_full_vs_reduced_silhouette.png", run_cfg)
else:
    print("Features, importances, or clustering_features not available.")

## Dead Feature Detector (D9.5)

Features with < 0.5% of total RF importance are "dead" — they contribute
almost nothing to regime classification. These are candidates for removal
from `clustering_features` in `settings.yaml`.

In [ ]:
if importances is not None and clustering_features:
    # Intersection: only look at features that are in clustering_features
    clust_imp = importances.reindex([f for f in clustering_features if f in importances.index]).dropna()

    if not clust_imp.empty:
        total = clust_imp.sum()
        pct = (clust_imp / total * 100).sort_values(ascending=True)
        dead_threshold = 0.5  # percent
        dead = pct[pct < dead_threshold]
        alive = pct[pct >= dead_threshold]

        print(f"Total clustering features in RF model: {len(clust_imp)}")
        print(f"Dead features (<{dead_threshold}% importance): {len(dead)}")
        print(f"Alive features (>={dead_threshold}% importance): {len(alive)}")

        if not dead.empty:
            print(f"\nDead features (candidates for removal):")
            for feat, val in dead.items():
                print(f"  {feat}: {val:.3f}%")

        # Visualization: horizontal bar chart of all features, dead ones in red
        fig, ax = plt.subplots(figsize=(10, max(4, len(pct) * 0.22)))
        colors = [plotting.CUSTOM_COLORS[1] if v < dead_threshold
                  else plotting.CUSTOM_COLORS[0] for v in pct.values]
        ax.barh(range(len(pct)), pct.values, color=colors, alpha=0.8)
        ax.set_yticks(range(len(pct)))
        ax.set_yticklabels(pct.index, fontsize=7)
        ax.axvline(dead_threshold, color="red", linestyle="--", linewidth=1,
                   alpha=0.6, label=f"{dead_threshold}% threshold")
        ax.set_xlabel("Importance (% of total)")
        ax.set_title(f"Feature Importance — Dead Feature Detection ({len(dead)} dead / {len(pct)} total)",
                     fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(axis="x", alpha=0.2)
        fig.tight_layout()
        plotting._save_or_show(fig, "11_dead_feature_detector.png", run_cfg)
    else:
        print("No clustering features found in RF model importances.")

    # Also flag features in clustering_features but NOT in RF model
    not_in_rf = [f for f in clustering_features if f not in importances.index]
    if not_in_rf:
        print(f"\nFeatures in clustering_features but NOT in RF model ({len(not_in_rf)}):")
        for f in not_in_rf:
            print(f"  {f}  (derivative-only — not used in supervised step)")
else:
    print("Importances or clustering_features not available.")